In [1]:
import kagglehub

path = kagglehub.dataset_download("joebeachcapital/restaurant-reviews")

print("Path to dataset files:", path)
import os

print(path)
print(os.listdir(path))

Using Colab cache for faster access to the 'restaurant-reviews' dataset.
Path to dataset files: /kaggle/input/restaurant-reviews
/kaggle/input/restaurant-reviews
['Restaurant reviews.csv']


In [2]:
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
df = pd.read_csv(path + "/Restaurant reviews.csv")

In [12]:
df.head()

,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures,7514,Clean_Review
0,Beyond Flavours,Rusha Chakraborty,"The ambience was good, food was quite good . h...",5,"1 Review , 2 Followers",5/25/2019 15:54,0,2447.0,ambienc good food quit good saturday lunch cos...
1,Beyond Flavours,Anusha Tirumalaneedi,Ambience is too good for a pleasant evening. S...,5,"3 Reviews , 2 Followers",5/25/2019 14:20,0,NaN,ambienc good pleasant even servic prompt food ...
2,Beyond Flavours,Ashok Shekhawat,A must try.. great food great ambience. Thnx f...,5,"2 Reviews , 3 Followers",5/24/2019 22:54,0,NaN,must tri great food great ambienc thnx servic ...
3,Beyond Flavours,Swapnil Sarkar,Soumen das and Arun was a great guy. Only beca...,5,"1 Review , 1 Follower",5/24/2019 22:11,0,NaN,soumen da arun great guy behavior sincereti go...
4,Beyond Flavours,Dileep,Food is good.we ordered Kodi drumsticks and ba...,5,"3 Reviews , 2 Followers",5/24/2019 21:37,0,NaN,food good order kodi drumstick basket mutton b...


In [38]:
ps = PorterStemmer()
stop_words = set(stopwords.words("english"))

def preprocess(text):

    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Tokenization
    text = text.split()

    # Remove stopwords
    text = [word for word in text if word not in stop_words]

    # Stemming
    text = [ps.stem(word) for word in text]

    return " ".join(text)

In [39]:
df["Clean_Review"] = df["Review"].apply(preprocess)

In [40]:
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

df = df.dropna(subset=["Rating"])

In [41]:
def sentiment(rating):

    if rating >= 4:
        return "Positive"

    elif rating == 3:
        return "Neutral"

    else:
        return "Negative"

df["Sentiment"] = df["Rating"].apply(sentiment)

In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df["Clean_Review"])

y = df["Sentiment"]

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [54]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train, y_train)

MultinomialNB()

In [55]:
y_pred = model.predict(X_test)

In [56]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print("Accuracy :", accuracy_score(y_test,y_pred))

print(confusion_matrix(y_test,y_pred))

print(classification_report(y_test,y_pred))

Accuracy : 0.8043151028600101
[[ 366    1  132]
 [  43    3  193]
 [  21    0 1234]]
              precision    recall  f1-score   support

    Negative       0.85      0.73      0.79       499
     Neutral       0.75      0.01      0.02       239
    Positive       0.79      0.98      0.88      1255

    accuracy                           0.80      1993
   macro avg       0.80      0.58      0.56      1993
weighted avg       0.80      0.80      0.75      1993



In [57]:
negative_reviews = df[df["Sentiment"]=="Negative"]["Clean_Review"]

text = " ".join(negative_reviews)

In [58]:
from collections import Counter

words = text.split()

counter = Counter(words)

print(counter.most_common(20))

[('order', 1535), ('food', 1530), ('place', 955), ('good', 867), ('tast', 710), ('servic', 700), ('chicken', 691), ('bad', 545), ('worst', 529), ('time', 516), ('restaur', 508), ('even', 503), ('like', 475), ('serv', 451), ('one', 444), ('biryani', 422), ('experi', 419), ('also', 319), ('qualiti', 316), ('disappoint', 298)]


In [59]:
print(df["Sentiment"].value_counts())

Sentiment
Positive    6274
Negative    2494
Neutral     1193
Name: count, dtype: int64


In [60]:
print(df["Rating"].mean())

3.601044071880333


In [61]:
restaurant_rating = df.groupby("Restaurant")["Rating"].mean()

print(restaurant_rating)

Restaurant
10 Downing Street                        3.80
13 Dhaba                                 3.48
3B's - Buddies, Bar & Barbecue           4.76
AB's - Absolute Barbecues                4.88
Absolute Sizzlers                        3.62
                                         ... 
Urban Asia - Kitchen & Bar               3.65
Yum Yum Tree - The Arabian Food Court    3.56
Zega - Sheraton Hyderabad Hotel          4.45
Zing's Northeast Kitchen                 3.65
eat.fit                                  3.20
Name: Rating, Length: 100, dtype: float64


In [62]:
top = restaurant_rating.sort_values(ascending=False)

print(top.head(10))

Restaurant
AB's - Absolute Barbecues                  4.88
B-Dubs                                     4.81
3B's - Buddies, Bar & Barbecue             4.76
Paradise                                   4.70
Flechazo                                   4.66
The Indi Grill                             4.60
Zega - Sheraton Hyderabad Hotel            4.45
Over The Moon Brew Company                 4.34
Beyond Flavours                            4.28
Cascade - Radisson Hyderabad Hitec City    4.26
Name: Rating, dtype: float64


In [63]:
bottom = restaurant_rating.sort_values()

print(bottom.head(10))

Restaurant
Hotel Zara Hi-Fi                         2.400
Asian Meal Box                           2.580
Pakwaan Grand                            2.710
Mathura Vilas                            2.820
Behrouz Biryani                          2.825
Shree Santosh Dhaba Family Restaurant    2.830
The Chocolate Room                       2.830
KFC                                      2.850
Club Rogue                               2.880
Desi Bytes                               2.900
Name: Rating, dtype: float64
